# Lesson 01 - Введение в AI-агентов

Добро пожаловать в первый урок курса **AI-агенты для начинающих**!

**AI-агент** — это программа, которая использует большую языковую модель (LLM) в качестве механизма рассуждения и может выполнять *действия* в реальном мире — вызывать API, запрашивать базы данных или запускать код — чтобы достичь цели от имени пользователя.

В этом ноутбуке вы создадите своего первого агента: **Туристического агента**, который рекомендует места для отдыха. По ходу вы узнаете, как:

1. Подключиться к Azure AI Foundry Agent Service с помощью **Microsoft Agent Framework**.
2. Дать агенту **инструмент** — простую функцию на Python, которую он может вызывать.
3. Запустить агента и проверить его ответ.
4. Воспроизводить ответ агента токен за токеном.


## Настройка

Перед запуском этой записной книжки убедитесь, что у вас есть:

1. **Проект Azure AI Foundry** с развернутой моделью для чата (например, `gpt-4o-mini`).
2. **Выполнен вход в Azure CLI** — выполните `az login` в вашем терминале.
3. **Установлены необходимые переменные окружения:**
   - `AZURE_AI_PROJECT_ENDPOINT` — конечная точка вашего проекта Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — имя развернутой модели.

В ячейке ниже устанавливаются необходимые вам пакеты Python.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Создание Вашего Первого Агента

Агенту нужны две вещи:

- **Инструкции**, которые говорят ему *кто он* и *как себя вести* (системный запрос).
- **Инструменты** — функции Python, декорированные с помощью `@tool`, которые агент может вызвать для получения информации или выполнения действий.

Ниже мы определяем простой инструмент, который возвращает список популярных туристических направлений. Агент будет использовать этот инструмент, когда пользователь попросит рекомендации по путешествиям.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Построчная передача ответа

Для более интерактивного взаимодействия вы можете **потоково** получать ответ агента. Вместо того чтобы ждать полного ответа, агент выдаёт части текста по мере их генерации. Это особенно полезно в чат-интерфейсах, где нужно отображать вывод в реальном времени.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Резюме

В этом уроке вы узнали, как:

- **Создать провайдера**, который подключается к службе Azure AI Foundry Agent Service через `AzureAIProjectAgentProvider`.
- **Определить инструмент** с помощью декоратора `@tool`, чтобы агент мог вызывать ваши функции на Python.
- **Запустить агента** с пользовательским сообщением и вывести его ответ.
- **Вести потоковую передачу ответов** для вывода в реальном времени.

В следующем уроке мы более подробно изучим агентские фреймворки и научимся давать агентам более мощные инструменты и возможности многошагового рассуждения.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:  
Этот документ был переведен с помощью сервиса автоматического перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия обеспечить точность, пожалуйста, учитывайте, что автоматические переводы могут содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется профессиональный перевод человеком. Мы не несем ответственности за любые недоразумения или неправильное толкование, возникшие в результате использования данного перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
